In [ ]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
API_KEY = userdata.get("NLPCaseStudy")
client = OpenAI(api_key = API_KEY)

In [ ]:
model = "gpt-4.1-nano-2025-04-14"
print("Client Initialized")

#### **Convert CSV to JSONL**

In [ ]:
import pandas as pd
import json

In [ ]:
SYSTEM_PROMPT = "You are a binary text classifier.Reply with only 0 or 1."

In [ ]:
def convert_csv_to_jsonl (csv_path : str,jsonl_path : str,
                          system_prompt = SYSTEM_PROMPT) -> None :
  df = pd.read_csv(csv_path)

  with open(jsonl_path,"w") as file :
    for _,row in df.iterrows() :
      record = {
          "messages" : [
              {"role" : "system","content" : system_prompt},
              {"role" : "user","content" : str(row["Abstract"])},
              {"role" : "assistant","content" : str(row["Label"])}
          ]
      }

      file.write(json.dumps(record) + "\n")

  print("JSONL File Saved to : ",jsonl_path)

#### **Upload Training and Validation File**

In [ ]:
def upload_file (file_path : str,label : str = "File") -> str :
  try :
    with open(file_path,"rb") as file :
      response = client.files.create(
          file = file,
          purpose = "fine-tune"
      )
      print(f"{label} file uploaded")
      print("File ID : ",response.id)
      return response.id

  except FileNotFoundError :
    print("File Not Found : ",file_path)
    raise

  except Exception as e :
    print("File Upload Failed : ",str(e))
    raise

#### **Create Fine-tuning Job**

In [ ]:
def fine_tuning_job (model : str,train_id : str,valid_id : str) -> str :
  try :
    job = client.fine_tuning.jobs.create(
        model = model,
        training_file = train_id,
        validation_file = valid_id
    )
    print("Fine-tuning Job Created")
    print("Job ID : ",job.id)
    print("Job Status : ",job.status)
    return job.id

  except Exception as e :
    print("Failed to Create Job",str(e))
    return None

#### **Monitor Fine-tuning Job**

In [ ]:
import time

In [ ]:
def monitor_job (job_id : str,poll_interval : int = 120) -> dict :
  TERMINAL_STATES = {"succeeded","failed","cancelled"}

  while True :
    job = client.fine_tuning.jobs.retrieve(job_id)
    status = job.status

    if status in TERMINAL_STATES :
      return {
          "Status" : job.status,
          "Trained Tokens" : job.trained_tokens,
          "Fine-tuned Model" : job.fine_tuned_model
      }

    print(f"Status : {status}, Next check in {poll_interval}s")
    time.sleep(poll_interval)

#### **Run Full Pipeline**

In [ ]:
def run_pipeline (csv_train : str,csv_valid : str,
                  model = str) -> dict :
  convert_csv_to_jsonl(csv_train,"train.jsonl")
  convert_csv_to_jsonl(csv_valid,"validation.jsonl")
  print()

  train_id = upload_file("train.jsonl",label = "Training")
  valid_id = upload_file("validation.jsonl",label = "Validation")
  print()

  job_id = fine_tuning_job(model,train_id,valid_id)
  print()

  result = monitor_job(job_id,)
  return result

### **Usage**

In [ ]:
result = run_pipeline(csv_train = "/content/train.csv",
                      csv_valid = "/content/validation.csv",
                      model = "gpt-4.1-nano-2025-04-14")
print(result)

### **Model Inference and Evaluation**

In [ ]:
FINE_TUNED_MODEL_ID = "ft:gpt-4.1-nano-2025-04-14:shoeb-sutar::DKpkctvW"

In [ ]:
def predict (model_id : str,text : str) :
  response = client.responses.create(
      model = model_id,
      input = [
          {"role" : "system","content" : SYSTEM_PROMPT},
          {"role" : "user","content" : text}
      ],
      temperature = 0
  )
  return int(response.output_text)

In [ ]:
x = predict(FINE_TUNED_MODEL_ID,"Abstract: A number of studies have been conducted to model the stock market indices using pure time series models or regression models based on macro economic variables. In this study, instead of focusing on modeling the actual levels of stock market indices I focus on predicting the direction (up/down) as investors who rely on technical analysis are more interested in the direction of stock market index than the actual prediction value. Therefore, in this study I look at best modelling approach for the direction prediction: time series (ARMA) or macro factor models or combination of both (ARDL). My study shows that macro factor models outperform for direction prediction as compared to ARMA or ARDL models. The study was performed on stock market direction prediction of stock indices of three South Asia countries: India, Pakistan and Malaysia. The macro economic factors that are considered for direction prediction are: Inflation, Unemployment and Exchange Rate monthly data from March 2016 to September 2021. Keywords: stock, equity, prediction, models, direction, ARMA, ARDL,macro")
x

In [ ]:
def run_inference_on_test (model_id : str,test_data : list) :
  predictions = []
  for abstract in test_data :
    response = client.responses.create(
        model = model_id,
        input = [
            {"role" : "system","content" : SYSTEM_PROMPT},
            {"role" : "user","content" : abstract}
        ],
        temperature = 0
    )
    predictions.append(int(response.output_text))
  return predictions

In [ ]:
import pandas as pd

In [ ]:
test_df = pd.read_csv("/content/test (2).csv")
test_df.head()

In [ ]:
# Convert all Abstarct into a List
test_data = list(test_df["Abstract"])
print("Test Data : \n",test_data)

In [ ]:
# True Labels for Comparison
true_labels = list(test_df["Label"])

In [ ]:
predictions = run_inference_on_test (FINE_TUNED_MODEL_ID,
                                     test_data)
print("Prediction on Test Data : \n",predictions)

### **Fine-Tuned Model Evaluation**

In [ ]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,
                             f1_score)
from sklearn.metrics import confusion_matrix,classification_report

In [ ]:
accuracy = accuracy_score(true_labels,predictions)
print("Accuracy on Test Data : ",accuracy)

precision = precision_score(true_labels,predictions)
print("Precision on Test Data : ",precision)

recall = recall_score(true_labels,predictions)
print("Recall on Test Data : ",recall)

f1 = f1_score(true_labels,predictions)
print("F1-Score on Test Data : ",f1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme()
plt.figure(figsize = (10,6))
cf = confusion_matrix(true_labels,predictions)
sns.heatmap(cf,cmap = "viridis",annot = True)
plt.title("Confusion Matrix (Fine-tuned Model)")

plt.show()

In [ ]:
cr = classification_report(true_labels,predictions)
print("Classification Report : \n",cr)